# ONCOVA 데모: 변이 목록 → 암 종류 후보 Top-3

- 목적: NGS 검사로 찾은 **유전자 변이 목록**을 넣으면, 암이 처음 생긴 장기 후보 3개와 확률, 근거 유전자를 보여주는 웹 화면을 만듦
- 사용 모델: 팀 최고 **v8A + 치환쌍** (11/12 리더보드 0.3546). `scripts/v8x.py v8A_PAIR PAIR EXPORT`로 저장한 모델 파일 하나를 씀
  - 1단계 XGBoost: 변이 있음/없음 + 변이 유전자 수 + 핫스팟 + 아미노산 치환쌍 비율
  - 2단계 로지스틱 회귀: 겹치는 라벨(신장암 묶음 KIPAN / 투명세포암 KIRC, 뇌종양 묶음 GBMLGG / 저등급 신경교종 LGG) 안에서 세부 아형을 정함
  - 학습할 때 같은 환자의 중복 기록을 한 명으로 정리함. test를 train과 대조하는 보정(형제 라벨)은 **쓰지 않음** → 실제 환자에게도 그대로 쓸 수 있는 모델
- 화면 도구: Gradio (실행하면 다른 사람에게 보낼 수 있는 임시 공유 링크가 나옴)
- 위치: 연구용 프로토타입이며 진단이 아님. 최종 판단은 의료진이 함 ([6_1 규제·윤리](./docs/6_1_규제윤리_박종부.md))

**데이터 사용 원칙**
- 이 노트북은 대회 데이터(train/test)를 불러오지 않음. 필요한 것은 모델 파일 하나에 모두 들어 있음
- 예시 환자는 잘 알려진 변이로 만든 **가상 환자**임. 대회 데이터의 환자는 공유 링크로 외부에 보여주면 안 되므로 쓰지 않음 (대회 규칙 제7조)

## 0. 설치 (처음 한 번만)

- 웹 화면을 만드는 `gradio`를 설치함
- 설치가 끝나면 **Kernel → Restart** 를 한 번 해야 다음 셀에서 불러올 수 있음

In [52]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.


## 1. 모델 불러오기

- `output/model_v8A_PAIR.pkl` 하나에 1단계 XGBoost, 2단계 로지스틱 회귀, 유전자 4,384개 순서, 핫스팟 목록, 치환쌍 목록, 아형 이름이 모두 들어 있음
- 파일이 없으면 `python scripts/v8x.py v8A_PAIR PAIR EXPORT`로 만듦 (1분 정도)
- `유전자 4384 / 핫스팟 … / 치환쌍 … / 아형 26`이 나오면 정상임

In [ ]:
import re                      # 변이 표기에서 숫자(위치)를 찾는 도구
import joblib                  # 저장한 모델 파일을 불러옴
import numpy as np             # 수치 계산
import pandas as pd            # 표 데이터 처리
import xgboost as xgb          # 근거 유전자(기여도) 계산에 사용
import scipy.sparse as sp      # 학습 때처럼 0을 '빈 칸'으로 넘기는 희소 행렬
import gradio as gr            # 웹 데모 화면

# v8x.py EXPORT로 저장한 모델 묶음
art = joblib.load('./output/model_v8A_PAIR.pkl')
gene_cols = art['gene_cols']                        # 유전자 4,384개 (학습 때 순서)
classes = art['classes']                            # 아형 26개 이름 (모델 출력 순서)
hotspots = sorted(art['hot_vocab'], key=art['hot_vocab'].get)   # 핫스팟 이름을 모델 열 순서대로
pair_idx = art['pair_idx']                          # 학습에 쓴 치환쌍 번호 (20×20 중 30명 이상에게 나온 것)
stage1 = art['stage1']                              # 1단계 XGBoost
stage2 = art['stage2']                              # 2단계 로지스틱 회귀 {(묶음, 세부): 모델}
gmask = art['gmask']                                # 2단계가 쓰는 유전자 (5명 이상에게 변이)
print('유전자', len(gene_cols), '/ 핫스팟', len(hotspots), '/ 치환쌍', len(pair_idx), '/ 아형', len(classes))

## 2. 피처 만드는 함수 (`scripts/v8x.py`와 같은 방식)

- 예측할 때도 **학습 때와 똑같은 방식**으로 피처를 만들어야 해서, `v8x.py`의 계산을 환자 한 명용으로 옮김
- `get_positions`: 변이 표기에서 위치 숫자를 뽑음 (`'E726K H1047R'` → `['726', '1047']`)
- `pair_ratio`: missense 변이마다 "원래 아미노산 → 바뀐 아미노산" 쌍을 세서, missense 수 + 1로 나눈 비율 (예: `V600E`는 V→E 한 번)
  - 암마다 돌연변이를 일으키는 원인(자외선, 흡연 등)이 달라서 많이 생기는 치환이 다름 → 흑색종·폐 편평세포암이 특히 좋아짐

In [ ]:
AA_ORDER = 'ACDEFGHIKLMNPQRSTVWY'                       # 아미노산 20개 (학습 때 순서)
MIS = re.compile(r'^([A-Z])(\d+)([A-Z])$')               # missense 표기 예: V600E

def pair_ratio(muts):
    pr = np.zeros(400, np.float32)                      # 20×20 치환쌍 개수
    n_mis = 0                                           # missense 개수
    for cell in muts.values():
        for t in cell.split():                          # 한 칸에 여러 변이가 있으면 공백으로 나뉨
            m = MIS.match(t)
            if not m or m.group(1) not in AA_ORDER or m.group(3) not in AA_ORDER:
                continue                                # missense가 아니면 건너뜀 (끊김, 밀림 등)
            a, b = AA_ORDER.index(m.group(1)), AA_ORDER.index(m.group(3))
            if a != b:                                  # 아미노산이 그대로인 변이(silent)는 제외
                pr[a * 20 + b] += 1
                n_mis += 1
    return (pr / (n_mis + 1))[pair_idx]                 # +1: 변이가 적으면 0 쪽으로 줄임, 학습에 쓴 쌍만

In [55]:
def get_positions(cell):
    # cell 예: 'V600E' 또는 'Q369* I368N' 또는 'WT'
    if cell == 'WT':                    # 변이가 없으면
        return []                       # 빈 리스트를 돌려줌
    positions = []                      # 위치 숫자를 모을 리스트
    for mut in cell.split():            # 공백 기준으로 나눔
        nums = re.findall(r'\d+', mut)  # 지금 보고 있는 변이 하나
        if nums:                        # 숫자가 하나라도 있으면
            positions.append(nums[0])   # 첫 번째 숫자 위치
    return positions

In [ ]:
def make_hotspot_features(muts):
    present = {f'{g}_{p}' for g, cell in muts.items() for p in get_positions(cell)}   # 이 환자의 '유전자_위치'
    return np.array([hs in present for hs in hotspots], np.float32)                  # 핫스팟마다 있으면 1

print(get_positions('E726K H1047R'))   # 확인용

## 3. 입력 글자 → 유전자별 변이

- 사용자가 `TP53 R175H, BRAF V600E`처럼 입력하면 `{'TP53': 'R175H', 'BRAF': 'V600E'}`로 바꿈
- 쉼표로 나누고, 첫 번째 띄어쓰기에서 유전자 이름과 변이 표기를 나눔
- 유전자 이름은 대문자로 통일함 (`braf` → `BRAF`)
- 같은 유전자가 두 번 나오면 대회 데이터처럼 공백으로 이어 붙임 (`PIK3CA E726K H1047R`)

In [57]:
def parse_input(text):
    muts = {}                                   # 유전자 → 변이 표기
    for item in text.split(","):                # 쉼표로 나눔
        item = item.strip()                     # 앞뒤 공백 제거
        if not item:                            # 빈 칸이면 건너뜀
            continue
        gene, variant = item.split(' ', 1)      # 'TP53 R175H' → 'TP53', 'R175H' (첫 공백에서 한 번만 나눔)
        gene = gene.upper()                     # 유전자 이름을 대문자로 통일 (tp53 → TP53)
        # 같은 유전자가 두 번 나오면 데이터처럼 공백으로 이어 붙임 (예: 'E726K H1047R')
        muts[gene] = (muts[gene] + ' ' + variant) if gene in muts else variant
    return muts

print(parse_input('TP53 R175H, braf V600E, PIK3CA E726K, PIK3CA H1047R'))

{'TP53': 'R175H', 'BRAF': 'V600E', 'PIK3CA': 'E726K H1047R'}


## 4. 예측 (Top-3)

- `build_features`: 입력하지 않은 유전자는 WT(변이 없음)로 두고, 0/1 + `n_mut` + 핫스팟 + 치환쌍 비율을 학습 때 순서로 붙임
- `predict_top3`: 두 단계로 예측함
  1. 1단계 XGBoost가 26개 아형 확률을 냄
  2. 겹치는 두 쌍은 확률을 합쳐 묶음으로 보고, 2단계 로지스틱 회귀가 "세부 아형일 확률"로 묶음 확률을 나눔 (예: 신장암 묶음 60% × KIRC일 확률 70% → KIRC 42%, KIPAN 18%)
  3. 1등은 리더보드 제출과 같은 규칙(묶음 먼저 고르고 세부는 50% 기준)으로 정하고, 나머지는 확률 순서
- 학습 데이터에 없는 유전자(예: KRAS)는 모델이 쓸 수 없으므로 따로 모아서 화면에 알려줌

In [ ]:
def build_features(muts):
    known = {g: v for g, v in muts.items() if g in gene_cols}             # 모델이 아는 유전자만
    binary = np.array([g in known for g in gene_cols], np.float32)        # 유전자별 변이 있음 1 / 없음 0
    n_mut = binary.sum()                                                  # 변이가 있는 유전자 수
    values = np.concatenate([binary, [n_mut], make_hotspot_features(known), pair_ratio(known)])
    names = list(gene_cols) + ['n_mut'] + hotspots + [f'pair_{i}' for i in pair_idx]
    return pd.DataFrame([values], columns=names)

def predict_top3(text):
    muts = parse_input(text)
    X = build_features(muts)
    p1 = stage1.predict_proba(sp.csr_matrix(X.to_numpy()))[0]             # 1단계: 26개 아형 확률 (학습과 같은 희소 행렬로 넣어야 결과가 같음)
    x2 = np.concatenate([X[gene_cols].to_numpy()[0][gmask], [np.log1p(X['n_mut'][0])]])[None]   # 2단계 입력
    proba, merged = p1.copy(), p1.copy()
    for (umb, spec), lr in stage2.items():
        group = p1[umb] + p1[spec]                                        # 묶음 전체 확률
        q = lr.predict_proba(x2)[0, 1]                                    # 묶음 안에서 세부 아형일 확률
        proba[spec], proba[umb] = group * q, group * (1 - q)              # 화면에 보여줄 확률
        merged[umb], merged[spec] = group, -1                             # 1등 정하기용 (제출과 같은 규칙)
    best = int(merged.argmax())
    for (umb, spec), lr in stage2.items():
        if best == umb:
            best = spec if lr.predict_proba(x2)[0, 1] >= 0.5 else umb
    top = [best] + [i for i in np.argsort(proba)[::-1] if i != best][:2]  # 1등은 제출 규칙, 2·3등은 확률 순
    result = [(classes[i], round(float(proba[i]) * 100, 1)) for i in top] # 확률을 %로 바꿈
    unknown = [g for g in muts if g not in gene_cols]                     # 데이터에 없는 유전자 (예: KRAS)
    return result, unknown, X, top

print(predict_top3('BRAF V600E')[:2])

## 5. 근거 유전자

- 1단계 XGBoost의 `pred_contribs` 기능으로 "이 환자를 1등 아형으로 고를 때 각 피처가 얼마나 밀어줬는지(기여도)"를 계산함
- 그중 **환자가 실제로 가진 변이**이면서 1등 아형 쪽으로 밀어준 것만, 큰 순서로 3개 뽑음
- `n_mut`(변이 개수)과 치환쌍 비율은 유전자가 아니라서 근거 목록에서는 뺌
- 주의: 근거 유전자는 "모델이 판단에 많이 참고한 변이"이지, 그 암의 **원인 유전자라는 뜻은 아님** ([3_3 모델 해석](./docs/3_3_모델_중요피처_혼동행렬.md))

In [ ]:
def evidence_genes(X, top_class, k=3):
    contrib = stage1.get_booster().predict(xgb.DMatrix(sp.csr_matrix(X.to_numpy())), pred_contribs=True)[0]   # (아형 26개, 피처 수 + 1)
    c = contrib[top_class, :-1]                    # 1등 아형에 대한 피처별 기여도 (마지막 칸은 기본값이라 뺌)
    names = np.array(X.columns)
    present = X.iloc[0].to_numpy() > 0             # 이 환자가 실제로 가진 변이만
    order = [i for i in np.argsort(-c)             # 기여도가 큰 순서로
             if present[i] and c[i] > 0 and names[i] != 'n_mut' and not names[i].startswith('pair_')][:k]
    return [names[i] for i in order]

result, unknown, X, top = predict_top3('IDH1 R132H, TP53 R273H, ATRX R1426*')
print(result, evidence_genes(X, top[0]))

## 6. 한글 암 이름과 가상 예시 환자

- `KOR`: 26개 아형 약어를 한글 이름으로 바꿔 보여줌 (새봄님 [1_1 문서](./docs/1_1_NGS_암아형_배경_김새봄.md) 표 기준). 일반 사용자도 결과를 읽을 수 있게 함
- `CASES`: 각 암에서 잘 알려진 변이로 만든 **가상 환자 8명**. 용어를 몰라도 고르기만 하면 써볼 수 있게 함
  - 예시 1~7은 모델이 떠올린 암을 맞힘
  - 예시 8 흑색종은 일부러 남긴 **어려운 예시**임. 흑색종은 실제로 변이가 수십~수백 개인 암이라, 4개만 넣으면 같은 BRAF V600E가 흔한 갑상선암으로 헷갈림 → 모델의 한계를 솔직하게 보여줌

In [60]:
# 26개 아형 한글 이름 (1_1 문서 표 기준) → 일반 사용자도 읽을 수 있게 함
KOR = {
    'ACC': '부신피질암', 'BLCA': '방광암', 'BRCA': '유방암', 'CESC': '자궁경부암',
    'COAD': '대장암', 'DLBC': '림프종', 'GBMLGG': '뇌종양(교모세포종·저등급 신경교종 묶음)',
    'HNSC': '두경부암', 'KIPAN': '신장암 묶음', 'KIRC': '신장 투명세포암', 'LAML': '급성 골수성 백혈병',
    'LGG': '저등급 신경교종(뇌종양)', 'LIHC': '간암', 'LUAD': '폐선암', 'LUSC': '폐 편평세포암',
    'OV': '난소암', 'PAAD': '췌장암', 'PCPG': '갈색세포종·부신경절종', 'PRAD': '전립선암',
    'SARC': '육종', 'SKCM': '피부 흑색종', 'STES': '위·식도암', 'TGCT': '고환암',
    'THCA': '갑상선암', 'THYM': '흉선종', 'UCEC': '자궁내막암',
}

# 가상 예시 환자 (실제 환자 아님): 각 암에서 잘 알려진 변이로 만든 예시
# ※ 대회 데이터(train/test)의 환자는 외부에 공유하면 안 되므로 쓰지 않음 (대회 규칙 제7조)
CASES = {
    '예시 1 · 갑상선암': ('BRAF V600E', 'THCA'),
    '예시 2 · 대장암': ('APC R1450*, TP53 R175H, PIK3CA E545K, FBXW7 R465H', 'COAD'),
    '예시 3 · 뇌종양(저등급 신경교종)': ('IDH1 R132H, TP53 R273H, ATRX R1426*', 'LGG'),
    '예시 4 · 신장암': ('VHL W117C', 'KIRC'),
    '예시 5 · 유방암': ('PIK3CA H1047R, CDH1 Q23*, GATA3 D336fs, MAP3K1 E1286*', 'BRCA'),
    '예시 6 · 급성 골수성 백혈병': ('NPM1 W288fs, IDH2 R140Q', 'LAML'),
    '예시 7 · 자궁내막암': ('PTEN R130G, PIK3CA H1047R, CTNNB1 S37F', 'UCEC'),
    '예시 8 · 흑색종 (어려운 예시)': ('BRAF V600E, CDKN2A R80*, PTEN R233*, TP53 P278S', 'SKCM'),
}
print(len(KOR), '개 아형 /', len(CASES), '개 예시')

26 개 아형 / 8 개 예시


## 7. 데모 화면 실행

**화면 구성**
- 맨 위: 작은 회색 "⚙️ 연구용 프로토타입" → 제목 → 목적 문구 ("암이 처음 생긴 곳의 후보 3개를 알려주는 참고용 도구")
- 왼쪽: ① 예시 고르기 → ② 예측하기 → ③ 결과 보기 순서 배지 (지금 단계만 검정 배경, `steps_html`) → 예시 환자 선택(▼ 안내) → 변이 목록 → [🔄 새로고침] [② 예측하기] → 접는 칸 "입력한 변이 읽는 법" (지금 입력한 변이를 하나씩 풀어서 설명, `explain`)
- 오른쪽: 맨 위 주황색 주의 박스(진단 아님) → 암 종류 후보 3개 표 → 💬 쉽게 말하면(근거가 된 입력 변이를 문장으로) → 예시와 일치 여부 → 근거 유전자
- 맨 위: 제목(아래쪽 테두리) + 오른쪽 위 **☀ Light / ☾ Dark** 알약 버튼(테마 전환)
- 꾸미기는 `CSS`에서 함: 좌우 칸 간격(`.main-row`), 새로고침 버튼 모양(`.reset-btn`), 제목 테두리(`.title-row`), 예측하기(`.predict-btn`), 테마 버튼(`.theme-btn`)
- 휴대폰(폭 640px 이하)에서는 `@media` 규칙으로 테마 버튼을 제목 위로 올리고 제목 글자를 줄임 (좁은 칸에 제목이 한 글자씩 꺾이던 문제 해결)
- 새로고침 아이콘: `assets/icon_refresh.svg` ([Icons8](https://icons8.com) "새롭게 하다" Plumpy)
- `gr.close_all()`이 이전 서버를 닫아서, 이 셀만 다시 실행해도 서버가 하나만 유지됨

**규제·윤리 반영** ([6_1](./docs/6_1_규제윤리_박종부.md))
- "진단" 대신 "예측 보조", "연구용 프로토타입(Research Use Only)" 표시
- 1등 확률이 40% 미만이면 **판단 보류** 를 권함 (`HOLD`)
- 근거 유전자 옆에 "원인 유전자라는 뜻은 아님" 주석

**일반인 피드백 반영 (11/10, 6명)** ([5_5](./docs/5_5_데모_피드백_질문지.md) 결과 정리)
- 처음 화면에서 무엇을 하는 도구인지 모름(4/6) → 목적 문구를 크게
- 사용 순서에서 헤맴(4/6), 드롭다운인지 몰랐음 → ①②③ 배지와 ▼ 안내
- 주의 문구를 신경 안 씀(4/6), "진단 프로그램"으로 이해(1/6) → 결과 맨 위 주황 박스로 옮김
- 변이 표기 설명이 어려움(5/6), BRAF 고정 예시 → 입력한 변이를 풀어서 설명
- 결과가 와닿지 않음 → "쉽게 말하면" 한 줄 추가

**실행 결과**
- `Running on public URL: https://….gradio.live` 주소를 보내면 다른 사람도 써볼 수 있음 (이 노트북이 켜져 있는 동안만 작동)
- 피드백 질문지: [5_5 데모 피드백 질문지](./docs/5_5_데모_피드백_질문지.md)

In [ ]:
HOLD = 40   # 1등 확률이 이보다 낮으면 "판단 보류" 표시 (%)

# 아미노산 한 글자 → 한글 이름 (변이 표기 풀이에 사용)
AA = {'A': '알라닌', 'R': '아르지닌', 'N': '아스파라진', 'D': '아스파르트산', 'C': '시스테인', 'E': '글루탐산', 'Q': '글루타민',
      'G': '글라이신', 'H': '히스티딘', 'I': '아이소류신', 'L': '류신', 'K': '라이신', 'M': '메싸이오닌', 'F': '페닐알라닌',
      'P': '프롤린', 'S': '세린', 'T': '트레오닌', 'W': '트립토판', 'Y': '타이로신', 'V': '발린'}

HOW_TO_RULE = """
- `V600E`: **600번째** 아미노산이 V에서 E로 **바뀜**
- `R1450*`: 1450번째에서 단백질이 **끊김** / `W288fs`: 288번째부터 읽는 틀이 **밀림**
- 여러 개는 쉼표로 구분합니다. 예: `BRAF V600E, TP53 R175H`
"""

def josa(word, pair):
    # 받침 유무로 조사 고르기: josa('글루탐산', '으로/로') → '글루탐산으로'
    with_b, without_b = pair.split('/')
    code = ord(word[-1]) - 0xAC00
    has_b = 0 <= code <= 11171 and code % 28 != 0
    if pair == '으로/로' and has_b and code % 28 == 8:     # ㄹ 받침은 '로' (예: 알코올로)
        return word + without_b
    return word + (with_b if has_b else without_b)

def explain_one(gene, token):
    # 변이 표기 하나를 쉬운 말로 풀이함 (모르는 형식이면 그대로 보여줌)
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', token)            # 예: V600E
    if m:
        a, pos, b = m.groups()
        if a == b:                                           # 예: L42L (아미노산이 그대로)
            return f'`{gene} {token}`: {pos}번째 아미노산 {josa(AA.get(a, a), "이/가")} **그대로** (단백질은 안 바뀌는 변이)'
        return f'`{gene} {token}`: {pos}번째 아미노산이 {AA.get(a, a)}에서 {josa(AA.get(b, b), "으로/로")} **바뀜**'
    pos = re.findall(r'\d+', token)
    if '*' in token and pos:
        return f'`{gene} {token}`: {pos[0]}번째에서 단백질이 **끊김**'
    if 'fs' in token and pos:
        return f'`{gene} {token}`: {pos[0]}번째부터 읽는 틀이 **밀림** (뒤쪽 단백질이 망가짐)'
    return f'`{gene} {token}`: 표기 형식이 특이해서 그대로 반영함'

def explain(text):
    # 입력칸이 바뀔 때마다, 지금 입력한 변이를 하나씩 풀어서 보여줌
    try:
        muts = parse_input(text) if text.strip() else {}
    except ValueError:
        muts = {}
    if not muts:
        return '예시를 고르거나 변이를 입력하면, 여기에서 **입력한 변이**를 하나씩 풀어서 설명합니다.' + HOW_TO_RULE
    lines = [explain_one(g, t) for g, cell in muts.items() for t in cell.split()]
    return '\n'.join(f'- {l}' for l in lines) + '\n\n**표기 규칙**' + HOW_TO_RULE

def steps_html(now):
    # 사용 순서 배지: 지금 단계(now)만 검정 배경, 지난 단계·남은 단계는 흰 배경
    names = ['① 예시 고르기', '② 예측하기', '③ 결과 보기']
    parts = []
    for i, name in enumerate(names, 1):
        cls = 'step active' if i == now else 'step'
        parts.append(f'<span class="{cls}">{name}</span>')
    return '<div class="steps">' + '<span class="step-arrow">→</span>'.join(parts) + '</div>'

def step_after_input(text):
    return steps_html(2 if text.strip() else 1)    # 변이가 채워지면 ② 예측하기 차례

START_MSG = '왼쪽에서 **① 예시 환자를 고르고** **② 예측하기** 를 누르면 여기에 결과가 나와요.'

def name_kor(code):
    return f'{code} {KOR.get(code, "")}'          # 'THCA' → 'THCA 갑상선암'

def run(text, case_name):
    if not text.strip():
        return START_MSG
    try:
        result, unknown, X, top = predict_top3(text)
    except ValueError:                             # 'BRAFV600E'처럼 띄어쓰기가 없으면 나눌 수 없음
        return '⚠️ 입력 형식을 확인해주세요. 유전자와 변이 사이에 띄어쓰기가 필요합니다. 예: `BRAF V600E`'
    genes = evidence_genes(X, top[0])
    # 후보 3개를 표로 보여줌
    table = ['| 순위 | 암 종류 | 확률 |', '|:---:|---|---:|']
    for rank, (code, pct) in enumerate(result, 1):
        table.append(f'| {rank} | **{code}** {KOR.get(code, "")} | {pct}% |')
    lines = ['<div class="warn-box">⚠️ <b>진단이 아닌 참고 정보예요.</b> 연구용 프로토타입이며, 최종 판단은 의료진이 합니다.</div>',
             '### ③ 결과: 암이 처음 생긴 곳 후보', '\n'.join(table)]
    # 쉬운 설명 한 줄: 근거 유전자에 해당하는 "입력한 변이"를 찾아서 문장으로 만듦
    muts = parse_input(text)
    used = []
    for g in genes:
        gene = g.split('_')[0]                       # 'BRAF_600' → 'BRAF'
        if gene in muts and f'{gene} {muts[gene]}' not in used:
            used.append(f'{gene} {muts[gene]}'.replace('*', '\\*'))   # '*'는 마크다운 굵게와 겹쳐서 글자로 표시
    top_name = KOR.get(result[0][0], result[0][0]).split('(')[0]
    if used:
        easy = f'💬 **쉽게 말하면**: 모델은 입력한 변이 중 **{", ".join(used)}** 를 보고 **{top_name}** 쪽으로 판단했어요.'
    else:
        easy = f'💬 **쉽게 말하면**: 판단에 쓸 만한 변이가 적어서, 변이 개수 같은 전체 모양으로 **{josa(top_name, "을/를")}** 골랐어요.'
    if result[0][1] < HOLD:
        lines.append('⚠️ 확신도가 낮아 **판단 보류** 를 권합니다.')
    lines.append(easy)
    # 가상 예시를 그대로 넣은 경우: 만들 때 떠올린 암과 비교해서 보여줌
    if case_name in CASES and CASES[case_name][0] == text.strip():
        expect = CASES[case_name][1]
        mark = '✅ 일치' if result[0][0] == expect else '❌ 다름'
        lines.append(f'**예시를 만들 때 떠올린 암**: {name_kor(expect)} → {mark}')
        if expect == 'SKCM':
            lines.append('※ 흑색종은 실제로 변이가 매우 많은 암이라(환자 1명당 수십~수백 개), 이렇게 몇 개만 넣으면 같은 BRAF V600E가 흔한 갑상선암으로 헷갈릴 수 있습니다.')
    lines.append(f'**근거 유전자**: {", ".join(genes) if genes else "없음"}')
    lines.append('※ 모델이 판단에 많이 참고한 변이이며, 암의 원인 유전자라는 뜻은 아닙니다.')
    if unknown:
        lines.append(f'※ 학습 데이터에 없는 유전자라 반영하지 않음: {", ".join(unknown)}')
    return '\n\n'.join(lines)                     # 빈 줄로 이어야 화면에서 줄이 나뉨

def fill_case(case_name):
    return CASES[case_name][0] if case_name in CASES else ''   # 예시를 고르면 입력칸을 채움

# 화면 꾸미기 (CSS)
CSS = """
/* 좌우 칸 간격 */
.main-row { gap: 48px !important; }

/* 예측하기: 어두운 회색 / 새로고침: 같은 계열 테두리 버튼 */
.predict-btn { background: #374151 !important; color: white !important; border: 1px solid #374151 !important; }
.predict-btn:hover { background: #1f2937 !important; }
.reset-btn { flex-wrap: nowrap !important; white-space: nowrap !important; padding-left: 10px !important; padding-right: 10px !important; background: white !important; color: #374151 !important; border: 1px solid #9ca3af !important; }
.reset-btn:hover { background: #f3f4f6 !important; }
.reset-btn, .predict-btn { height: 44px !important; min-height: 44px !important; }   /* 두 버튼 높이를 같게 */
.reset-btn img { width: 16px !important; height: 16px !important; margin-right: 8px !important; opacity: 0.8; }   /* 아이콘과 글자 사이 띄우기 */
.dark .reset-btn img { filter: invert(1); }   /* 다크 모드에서는 아이콘을 흰색으로 */
.dark .predict-btn { background: #4b5563 !important; border-color: #4b5563 !important; }
.dark .predict-btn:hover { background: #6b7280 !important; }
.dark .reset-btn { background: transparent !important; color: #e5e7eb !important; border-color: #6b7280 !important; }
.dark .reset-btn:hover { background: rgba(255, 255, 255, 0.06) !important; }

/* 제목 줄: 아래쪽 테두리 */
.title-row { border-bottom: 1px solid var(--border-color-primary); padding-bottom: 12px; margin-bottom: 8px; align-items: center; }

/* 오른쪽 위 Light / Dark 알약 버튼 (감싸는 테두리 없음) */
.theme-row { gap: 8px !important; flex-wrap: nowrap !important; justify-content: flex-end; background: none !important; border: none !important; }
.theme-btn { border-radius: 999px !important; border: 1px solid #e5e7eb !important; background: white !important;
             color: #9ca3af !important; font-weight: 500 !important; box-shadow: none !important; min-width: 92px !important; }
.theme-btn.active { color: #374151 !important; border-color: #9ca3af !important; }
.dark .theme-btn { background: transparent !important; border-color: #4b5563 !important; color: #6b7280 !important; }
.dark .theme-btn.active { color: #e5e7eb !important; border-color: #9ca3af !important; }

/* 맨 위 목적 문구: 크게 */
.lead p { font-size: 1.0rem !important; line-height: 1.6 !important; word-break: keep-all; }   /* 단어 중간에서 줄이 꺾이지 않게 */
.lead h1 { margin-top: 0 !important; }
.lead em.kicker { font-style: normal; color: #696969; font-size: 0.9rem; }   /* "연구용 프로토타입" 작은 회색 글씨 */
.dark .lead em.kicker { color: #9ca3af; }
/* 사용 순서 ①②③ 배지 */
.steps { display: flex; gap: 8px; flex-wrap: wrap; align-items: center; margin: 4px 0 12px; }
.step { background: white; color: #9ca3af; border: 1px solid #e5e7eb; border-radius: 999px; padding: 6px 14px; font-weight: 600; font-size: 0.95rem; }
.step.active { background: #101E30; color: white; border-color: #101E30; }   /* 지금 단계만 검정 */
.step.active, .step.active * { color: white !important; }   /* 번역기·테마가 글자를 감싸도 흰 글자 유지 */
.step-arrow { color: #9ca3af; }
/* 결과 맨 위 주의 박스 */
.warn-box { background: #FFF4E5; border: 1px solid #F5A524; color: #7A4B00; border-radius: 10px; padding: 10px 14px; font-size: 0.95rem; }
.dark .warn-box { background: rgba(245, 165, 36, 0.12); color: #FCD9A0; }
.dark .step { background: transparent; color: #6b7280; border-color: #4b5563; }
.dark .step.active { background: #e5e7eb; color: #101E30; border-color: #e5e7eb; }
.dark .step.active, .dark .step.active * { color: #101E30 !important; }

/* 휴대폰 화면 (폭 640px 이하): 제목이 좁게 눌리지 않도록 테마 버튼을 제목 위로 올리고 글자를 줄임 */
@media (max-width: 640px) {
    .title-row { flex-direction: column-reverse !important; align-items: stretch !important; gap: 4px !important; }
    .title-row > * { min-width: 100% !important; }
    .title-row h1 { font-size: 1.5rem !important; line-height: 1.3 !important; word-break: keep-all; }   /* 단어 중간에서 줄이 꺾이지 않게 */
    .theme-row { justify-content: flex-end !important; }
    .theme-btn { min-width: 72px !important; flex: 0 0 auto !important; }   /* 버튼이 화면 폭만큼 늘어나지 않게 */
    .main-row { gap: 24px !important; }
    table th:first-child, table td:first-child, table th:last-child, table td:last-child { white-space: nowrap; }   /* 순위·확률이 두 줄로 꺾이지 않게 */
    .steps { flex-wrap: nowrap !important; gap: 4px; }                 /* 배지 3개를 한 줄에 */
    .step { padding: 4px 9px; font-size: 0.78rem; white-space: nowrap; }
    .step-arrow { font-size: 0.75rem; }
    .reset-btn img { margin-right: 5px !important; }
    .reset-btn { min-width: 96px !important; }   /* 좁은 폰에서도 글자가 테두리에 붙지 않게 */   /* 좁은 화면에서도 아이콘과 글자를 한 줄에 */
}
"""

# Light / Dark 전환: Gradio는 화면 맨 바깥(body)에 'dark' 클래스가 있으면 다크 테마로 그림
def theme_js(mode):
    return f"""
    () => {{
        if ('{mode}' === 'dark') {{ document.body.classList.add('dark'); }}
        else {{ document.body.classList.remove('dark'); }}
        document.querySelectorAll('.theme-btn').forEach(b => b.classList.remove('active'));
        document.querySelector('#{mode}-btn').classList.add('active');   // 지금 테마 버튼만 진하게
    }}
    """

gr.close_all()                                    # 이전에 띄운 데모 서버가 있으면 닫음 (여러 번 실행해도 하나만 유지)

with gr.Blocks(title='ONCOVA 데모') as demo:
    with gr.Row(elem_classes='title-row'):                  # 제목 줄 (아래쪽 테두리) + 오른쪽 테마 전환
        gr.Markdown('<em class="kicker">⚙️&nbsp; 연구용 프로토타입</em>\n\n'      # 제목 위 작은 회색 글씨
                    '# ONCOVA 암 종류 예측 보조\n'
                    '유전자 검사(NGS)에서 찾은 **변이 목록**을 넣으면,<br>'
                    '**암이 처음 생긴 곳의 후보 3개**를 알려주는 **참고용 도구**예요.',
                    elem_classes='lead')
        with gr.Row(elem_classes='theme-row', scale=0):        # 감싸는 테두리 없이 버튼만 나란히
            light_btn = gr.Button('☀ Light', size='sm', elem_id='light-btn', elem_classes='theme-btn')
            dark_btn = gr.Button('☾ Dark', size='sm', elem_id='dark-btn', elem_classes='theme-btn')
    with gr.Row(elem_classes='main-row'):                   # 왼쪽 입력 / 오른쪽 결과
        with gr.Column():
            steps = gr.HTML(steps_html(1))                       # 처음엔 ① 단계
            case = gr.Dropdown(list(CASES), value=None, label='① 예시 환자 고르기 (가상 환자)',
                               info='▼ 칸을 누르면 예시 목록이 나와요')        # 처음엔 아무것도 안 고른 상태
            text = gr.Textbox(label='변이 목록 (예시를 고르면 자동으로 채워져요 · 직접 입력도 가능)', lines=3,
                              placeholder='예: BRAF V600E, TP53 R175H')
            with gr.Row():                                  # 버튼 두 개를 나란히
                reset = gr.Button('새로고침', icon='assets/icon_refresh.svg', elem_classes='reset-btn', scale=1, min_width=0)   # 폭 1/4, 아이콘: Icons8
                btn = gr.Button('② 예측하기', elem_classes='predict-btn', scale=3, min_width=0)     # 폭 3/4
            with gr.Accordion('💡 입력한 변이 읽는 법 (눌러서 펼치기)', open=False):
                how = gr.Markdown(explain(''))                      # 입력칸이 바뀌면 같이 바뀜
        with gr.Column():
            out = gr.Markdown(START_MSG)
    case.change(fill_case, inputs=case, outputs=text)                 # 예시를 고르면 입력칸 채우기
    text.change(explain, inputs=text, outputs=how)                    # 입력한 변이 풀이를 바로 갱신
    text.change(step_after_input, inputs=text, outputs=steps)         # 변이가 채워지면 ② 단계로
    btn.click(lambda: steps_html(3), outputs=steps)                   # 예측하면 ③ 단계로
    btn.click(run, inputs=[text, case], outputs=out)                 # 예측하기
    reset.click(lambda: (None, '', START_MSG, steps_html(1)), outputs=[case, text, out, steps])   # 예시·입력·결과·단계를 처음 상태로
    light_btn.click(None, js=theme_js('light'))                      # Light / Dark 전환 (브라우저에서 바로 바뀜)
    dark_btn.click(None, js=theme_js('dark'))
    demo.load(None, js=theme_js('light'))                             # 처음에는 Light로 시작

demo.launch(share=True, css=CSS)


Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://57144745832cdb1fc0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
